### Imports and R Environment Setup


In [82]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [83]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
import rpy2.robjects as ro
from rpy2.robjects import numpy2ri
from rpy2.robjects.conversion import localconverter

# Your updated package imports
from graphical_sampling.sampling import KMeansSampler
from graphical_sampling.population import Population
from package_sampling.utils import inclusion_probabilities

# Initialize the converter
numpy2ri_converter = numpy2ri.converter
conv = ro.default_converter + numpy2ri_converter

# Initialize R with updated scoring functions
ro.r("""
    library(BalancedSampling)
    library(sampling)
    library(WaveSampling)

    calc_r_metrics <- function(coords, probs, sample_idx) {
        coords_mat <- as.matrix(coords)
        probs_vec <- as.numeric(probs)
        
        # Note: indices in R are 1-based. 
        # sample_idx should be adjusted before passing or inside here.
        
        # Spatial Balance (SB)
        sb_val <- tryCatch(sb(probs_vec, coords_mat, sample_idx), error = function(e) Inf)
        
        # Spatial Balance Local (SBLB)
        sblb_val <- tryCatch(sblb(probs_vec, coords_mat, sample_idx), error = function(e) Inf)
        
        # Moran's I (IB) - requires weight matrix W
        W0 <- wpik(coords_mat, probs_vec)
        W <- W0 - diag(diag(W0))
        
        # Create a binary mask for the sample
        sample_mask <- rep(0, length(probs_vec))
        sample_mask[sample_idx] <- 1
        
        ib_val <- tryCatch(IB(W, sample_mask), error = function(e) Inf)
        
        return(c(ib = ib_val, sb = sb_val, sblb = sblb_val))
    }
""")

def evaluate_sampler(sampler: KMeansSampler):
    """
    Evaluates the KMeansSampler using both internal Python properties 
    and external R metrics.
    """
    # 1. Get all possible samples and their probabilities from the joint design
    all_samples = sampler.all_samples       # Array of shape (M, n)
    all_probs = sampler.all_samples_probs   # Array of shape (M,)
    
    results = []
    
    # Use tqdm for progress tracking
    for i in tqdm(range(len(all_samples)), desc="Evaluating Samples"):
        sample_indices = all_samples[i]
        # Adjust to 1-based indexing for R
        r_sample_idx = sample_indices + 1
        
        with localconverter(conv):
            # Pass data to R
            r_metrics = ro.r['calc_r_metrics'](
                sampler.coords, 
                sampler.probs, 
                r_sample_idx
            )
            r_metrics_np = np.array(r_metrics)
        
        # Python-side Density score
        # Using the internal cached property logic
        density_score = sampler.density_scores[i]
        
        results.append({
            'sample_id': i,
            'probability': all_probs[i],
            'density': density_score,
            'moran_i': r_metrics_np[0],
            'voronoi_sb': r_metrics_np[1],
            'local_balance': r_metrics_np[2]
        })
    
    df_results = pd.DataFrame(results)
    
    # Calculate Expected Values (Weighted Averages)
    summary = {
        'Exp_Density': np.sum(df_results['density'] * df_results['probability']),
        'Exp_Moran': np.sum(df_results['moran_i'] * df_results['probability']),
        'Exp_SB': np.sum(df_results['voronoi_sb'] * df_results['probability']),
        'Exp_LocalBalance': np.sum(df_results['local_balance'] * df_results['probability'])
    }
    
    return df_results, summary



### Optimized Sampling Functions

In [84]:
import os
import numpy as np
import pandas as pd
import rpy2.robjects as ro
from tqdm import tqdm
from rpy2.robjects import numpy2ri, pandas2ri
from rpy2.robjects.conversion import localconverter

# Define the converter context for modern rpy2
combined_converter = ro.default_converter + numpy2ri.converter + pandas2ri.converter

def run_sampling_design(method, coords, probs, n, num_samples, is_EP):
    N = len(coords)
    
    # 1. Python Methods (Nmcs and Rand)
    if method == "Nmcs":
        pop = Population(coords=coords, probs=probs)
        sampler = KMeansSampler(
            population=pop, 
            n=n, 
            n_zones=(2, 2), 
            zone_builder="sweep"
        )
        return sampler.sample(num_samples)

    if method == "Rand":
        samples_idx = np.zeros((num_samples, n), dtype=int)
        for i in range(num_samples):
            samples_idx[i] = np.random.choice(N, n, replace=False)
        return samples_idx

    # 2. R Methods (Lopi, Wave, Maxe, SCP)
    samples_idx = np.zeros((num_samples, n), dtype=int)
    
    with localconverter(combined_converter):
        ro.globalenv['coords_r'] = coords
        ro.globalenv['probs_r'] = probs
        
        # Load the core balanced sampling library
        ro.r("library(BalancedSampling)")
        
        for i in range(num_samples):
            if method == "Lopi":
                # Local Pivotal Method 2
                samples_idx[i] = np.array(ro.r("lpm2(probs_r, coords_r)")) - 1
                
            elif method == "SCP":
                # Spatially Correlated Poisson Sampling
                # scps returns 1-based indices
                samples_idx[i] = np.array(ro.r("scps(probs_r, coords_r)")) - 1
                
            elif method == "Wave":
                ro.r("library(WaveSampling)")
                mask = ro.r("wave(coords_r, probs_r)")
                samples_idx[i] = np.where(np.array(mask).astype(bool))[0]
                
            elif method == "Maxe":
                mask = ro.r("sampling::UPmaxentropy(probs_r)")
                samples_idx[i] = np.where(np.array(mask).astype(bool))[0]

    return samples_idx

In [85]:
def calculate_ht_estimator(y, sample_indices, probs):
    """
    Calculates the Horvitz-Thompson estimator for the population total.
    """
    # Filter for valid indices (stripping -1 placeholders from empty clusters)
    valid_mask = sample_indices >= 0
    valid_idx = sample_indices[valid_mask]
    
    sample_y = y[valid_idx]
    sample_probs = probs[valid_idx]
    
    # Horvitz-Thompson Total Estimate = sum(y_i / pi_i)
    return np.sum(sample_y / sample_probs)

### Metrics and Spread Calculation

In [86]:
def calculate_all_scores(coords, probs, sample_idx, n, N, density_measure, y_val):
    """
    Calculates various spatial and statistical scores for a given sample.
    
    Args:
        coords: Population coordinates.
        probs: Inclusion probabilities.
        sample_idx: 1D array of selected unit indices.
        n: Sample size.
        N: Population size.
        density_measure: An instance of the Density class.
        y_val: The variable of interest for HT estimation.
    """
    # 1. Density Score (Python)
    # The score method expects a 2D array of shape (n_samples, n)
    dens_score = density_measure.score(sample_idx.reshape(1, -1))
    
    # 2. HT Estimator Total
    # Uses the formula: Sum(y_i / pi_i)
    ht_val = np.sum(y_val[sample_idx] / probs[sample_idx])
    
    # 3. R Metrics (Spatial Balance)
    # Preparing data for the R environment
    with localconverter(conv):
        ro.globalenv['coords'] = coords
        ro.globalenv['probs'] = probs
        # R uses 1-based indexing for sample indices
        ro.globalenv['s_idx_r'] = sample_idx + 1 
        
        # Note: Your R function calc_r_metrics now takes (coords, probs, sample_idx)
        # We pass the R-adjusted 1-based indices.
        r_results = ro.r("calc_r_metrics(coords, probs, s_idx_r)")
        r_results_np = np.array(r_results)

    # Return Order: Density, Voronoi (sb), Moran (ib), Local Balance (sblb), HT_Total
    # r_results indices based on R function: c(ib, sb, sblb)
    return (
        dens_score[0],     # Density
        r_results_np[1],   # Voronoi (sb)
        r_results_np[0],   # Moran (ib)
        r_results_np[2],   # Local Balance (sblb)
        ht_val             # HT Estimator Total
    )

### The Main Execution Loop

In [88]:
# --- Main Configuration and Loop ---
folder = "/config/ws/graphical-sampling/populations"
results_folder = "data_samples"
pop_names = ["random_uneq"]
sample_cnt = 1000 
n_size = 5

os.makedirs(results_folder, exist_ok=True)

for name in pop_names:
    # 1. Load and Prep Data
    file_path = os.path.join(folder, f"{name}_N=100.csv")
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        continue
        
    df = pd.read_csv(file_path)
    coords = df[["x", "y"]].values.astype(float)
    y_values = df["y"].values
    true_sum_y = np.sum(y_values) 

    # Calculate inclusion probabilities
    pik = inclusion_probabilities(df["prob"].values, n_size)
    N = len(pik)
    n = int(np.round(np.sum(pik)))
    is_EP = np.allclose(pik, pik[0])

    print(f"\n--- Processing {name} (N={N}, n={n}, True Total={true_sum_y:.3f}) ---")

    # 2. Setup Metrics
    pop_wrapped = Population(coords=coords, probs=pik)
    density_measure = Density(population=pop_wrapped, k=n, n_jobs=-1)

    # 3. Sampling and Scoring
    all_data = []
    methods = ["Nmcs", "Wave", "Lopi", "SCP", "Maxe", "Rand"] 

    for m in methods:
        print(f"Running {m}...")
        samples = run_sampling_design(m, coords, pik, n, sample_cnt, is_EP)
        
        for i in tqdm(range(sample_cnt), desc=f"Scoring {m}"):
            s_idx = samples[i]
            valid_s_idx = s_idx[s_idx >= 0]
            
            # Use HT for all methods, but for SRS (Rand), we will compute the correct N * mean(y_s)
            if m == "Rand":
                # SRS Estimator: N * y_bar
                est_total = N * np.mean(y_values[valid_s_idx])
                # We still need the spatial scores
                metrics = list(calculate_all_scores(coords, pik, valid_s_idx, n, N, density_measure, y_values))
                metrics[-1] = est_total # Replace HT with SRS result for Rand
            else:
                metrics = calculate_all_scores(coords, pik, valid_s_idx, n, N, density_measure, y_values)
            
            all_data.append([m] + list(metrics))

    # 4. Results Processing
    res_df = pd.DataFrame(all_data, columns=["Method", "D", "V", "M", "L", "HT"])
    
    # 5. Summary Statistics Generation
    summary = res_df.groupby("Method").agg({
        "D": ["mean", "std"], 
        "V": ["mean", "std"], 
        "M": ["mean", "std"], 
        "L": ["mean", "std"], 
        "HT": ["mean", "var"]
    })

    # Rename to requested abbreviations
    summary.columns = ["Dm", "Ds", "Vm", "Vs", "Mm", "Ms", "Lm", "Ls", "HTm", "HTv"]

    # Calculate Relative Bias (RB): (Mean_Est - TrueTotal) / TrueTotal
    summary["RB"] = (summary["HTm"] - true_sum_y) / true_sum_y

    # Calculate Efficiency (Eff) relative to SRS (Rand)
    if "Rand" in summary.index:
        rand_var = summary.loc["Rand", "HTv"]
        summary["Eff"] = rand_var / summary["HTv"].replace(0, np.nan)
    
    # Select and order requested columns
    final_cols = ["RB", "Eff", "Dm", "Ds", "Vm", "Vs", "Mm", "Ms", "Lm", "Ls"]
    summary = summary[final_cols].round(3)

    print("\nSimulation Summary:")
    print(summary)
    
    # Save the detailed and summary data
    res_df.to_csv(os.path.join(results_folder, f"samples_{name}.csv"), index=False)
    summary.to_csv(os.path.join(results_folder, f"summary_{name}.csv"))


--- Processing random_uneq (N=100, n=5, True Total=50.913) ---
Running Nmcs...


Scoring Nmcs:   0%|          | 0/1000 [00:00<?, ?it/s]

Scoring Nmcs: 100%|██████████| 1000/1000 [00:14<00:00, 67.08it/s]


Running Wave...


Scoring Wave: 100%|██████████| 1000/1000 [00:14<00:00, 66.73it/s]


...

Running Maxe...


Scoring Maxe: 100%|██████████| 1000/1000 [00:15<00:00, 66.43it/s]


Running Rand...


Scoring Rand: 100%|██████████| 1000/1000 [00:14<00:00, 70.58it/s]


Simulation Summary:
           RB    Eff     Dm     Ds     Vm     Vs     Mm     Ms     Lm      Ls
Method                                                                       
Lopi   -0.027  0.286 -0.157  0.270  0.127  0.087 -0.205  0.102  0.635   0.771
Maxe    0.039  0.017 -0.347  0.292  0.262  0.197 -0.041  0.137  0.745   2.567
Nmcs   -0.032  0.266 -0.129  0.244  0.126  0.076 -0.172  0.117  0.614   0.672
Rand   -0.010  1.000 -0.183  0.355  0.505  0.365 -0.059  0.117  6.042  16.691
SCP     0.002  0.206 -0.140  0.269  0.112  0.084 -0.241  0.094  0.637   0.805
Wave    0.035  0.017 -0.111  0.237  0.104  0.072 -0.317  0.090  0.667   2.588
